In [1]:
from datasets import load_dataset
ds = load_dataset("timm/oxford-iiit-pet")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [2]:
# Inspect dataset features
print(f"Dataset features: {ds['train'].features}")
print(f"Example item keys: {ds['train'][0].keys()}")

Dataset features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle', 'bengal', 'birman', 'bombay', 'boxer', 'british_shorthair', 'chihuahua', 'egyptian_mau', 'english_cocker_spaniel', 'english_setter', 'german_shorthaired', 'great_pyrenees', 'havanese', 'japanese_chin', 'keeshond', 'leonberger', 'maine_coon', 'miniature_pinscher', 'newfoundland', 'persian', 'pomeranian', 'pug', 'ragdoll', 'russian_blue', 'saint_bernard', 'samoyed', 'scottish_terrier', 'shiba_inu', 'siamese', 'sphynx', 'staffordshire_bull_terrier', 'wheaten_terrier', 'yorkshire_terrier']), 'image_id': Value('string'), 'label_cat_dog': ClassLabel(names=['cat', 'dog'])}
Example item keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])


In [3]:
import os
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Data processing and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# PyTorch and deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.models as models

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report
)

# Formatting
from tabulate import tabulate

# ==================== DEVICE AND GPU SETUP ====================
print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# GPU optimizations (for T4 or similar)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

# ==================== RANDOM SEED FOR REPRODUCIBILITY ====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch: 2.10.0+cpu
device: cpu


In [ ]:
import torchvision.transforms as T

# Inspect features to be sure
print(f"Available keys: {ds['train'].features.keys()}")

# Classification Dataset (no mask)
class PetClassificationDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert("RGB")
        label = item['label']
        if self.transform:
            image = self.transform(image)
        return image, label

# Define transforms
image_transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Prepare data loaders
train_ds = PetClassificationDataset(ds['train'], transform=image_transform)
test_ds = PetClassificationDataset(ds['test'], transform=image_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print("Classification dataset initialized.")

Available keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])
Dataset initialized. Using mask key: segmentation_mask


In [8]:
# ==================== EDA: Visualize Images and Labels ====================
import matplotlib.pyplot as plt

def show_image_and_label(dataset, idx):
    image, label = dataset[idx]
    # Undo normalization for visualization
    img = image.numpy().transpose(1, 2, 0)
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(f'Label: {label}')
    plt.axis('off')
    plt.show()

# Show a few samples from the training set
for i in range(3):
    show_image_and_label(train_ds, i)

# ==================== EDA: Label Distribution ====================
all_labels = [train_ds[i][1] for i in range(len(train_ds))]
plt.hist(all_labels, bins=np.arange(-0.5, max(all_labels)+1.5, 1), rwidth=0.8)
plt.title('Label Distribution (train set)')
plt.xlabel('Class label')
plt.ylabel('Count')
plt.show()

KeyError: 'segmentation_mask'

# Model Definition, Training, and Evaluation
We will use a pre-trained ResNet18 model for classification, train it, and evaluate its performance.

In [ ]:
# ==================== Model Definition ====================
import torchvision.models as models

num_classes = len(set([train_ds[i][1] for i in range(len(train_ds))]))
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# ==================== Loss and Optimizer ====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# ==================== Training Loop ====================
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

num_epochs = 5
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
# ==================== Evaluation: Classification Report ====================
from sklearn.metrics import classification_report

def get_all_preds(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return all_labels, all_preds

all_labels, all_preds = get_all_preds(model, test_loader, device)
print(classification_report(all_labels, all_preds))

In [ ]:
# ==================== Evaluation: Confusion Matrix ====================
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [6]:
# Debug: Check available keys in a sample item
print('Sample train item keys:', ds['train'][0].keys())
print('Sample test item keys:', ds['test'][0].keys())

Sample train item keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])
Sample test item keys: dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])
